In [1]:
import pickle
import pandas as pd
import sqlite3

In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

### Régression Logistique

In [6]:
conn = sqlite3.connect("../DATA/meteo.db")

df = pd.read_sql("SELECT * FROM meteo", conn)

X = df.drop(["pluie", "pluie_1h"], axis = 1) #Les features
y = df["pluie"] # La cible

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(class_weight='balanced')
model.fit(X_train_scaled, y_train)
y_pred_test = model.predict(X_test_scaled)
y_pred_train = model.predict(X_train_scaled)

precision_train = precision_score(y_train, y_pred_train)
precision_test = precision_score(y_test, y_pred_test)

In [7]:
# 1. Comparer les scores
from sklearn.metrics import accuracy_score, recall_score, f1_score

print(f"Precision train: {precision_train:.3f}")
print(f"Precision test: {precision_test:.3f}")
print(f"Accuracy train: {accuracy_score(y_train, y_pred_train):.3f}")
print(f"Accuracy test: {accuracy_score(y_test, y_pred_test):.3f}")

# 2. Utiliser la validation croisée
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='precision')
print(f"CV Precision: {scores.mean():.3f} (+/- {scores.std():.3f})")

# 3. Analyser les coefficients
import numpy as np
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': np.abs(model.coef_[0])
}).sort_values('coefficient', ascending=False)
print(coef_df)


Precision train: 0.269
Precision test: 0.257
Accuracy train: 0.814
Accuracy test: 0.814
CV Precision: 0.268 (+/- 0.010)
          feature  coefficient
1     temperature     2.703205
4  point_de_rosee     2.433559
3        humidite     2.266359
6      vent_moyen     0.799688
5      visibilite     0.494548
2        pression     0.463197
9           heure     0.385957
7            mois     0.128807
0              id     0.116598
8            jour     0.026802


In [4]:
precision_test

1.0

## Random Forest

In [16]:
conn = sqlite3.connect("../DATA/meteo.db")

df = pd.read_sql("SELECT * FROM meteo", conn)

X = df.drop(["pluie", "pluie_1h"], axis = 1) #Les features
y = df["pluie"] # La cible

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(class_weight='balanced')
model.fit(X_train_scaled, y_train)
y_pred_test = model.predict(X_test_scaled)
y_pred_train = model.predict(X_train_scaled)

precision_train = precision_score(y_train, y_pred_train)
precision_test = precision_score(y_test, y_pred_test)

In [17]:
print(f"Précision entraînement: {precision_score(y_train, y_pred_train):.3f}")
print(f"Précision test: {precision_score(y_test, y_pred_test):.3f}")

Précision entraînement: 1.000
Précision test: 0.859


In [38]:
precision_train

1.0

In [39]:
precision_test

1.0

In [33]:
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred) 
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)


In [28]:
y_test

4622    0
1530    0
8706    0
2233    0
676     0
       ..
6820    0
802     0
8709    0
6338    0
2068    0
Name: pluie, Length: 4498, dtype: int64

In [34]:
precision

1.0

In [35]:
recall

1.0

Le modèle de Random Forest a de meilleurs résultats lors de l'évaluation.

In [44]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    #'max_features': ['auto', 'sqrt']
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    error_score='raise'
)

grid_search.fit(X_train_scaled, y_train)
best_model = grid_search.best_estimator_

# Évaluer le meilleur modèle
y_pred_test = best_model.predict(X_test_scaled)

KeyboardInterrupt: 